# Digit Recognizer — CNN on MNIST

MNIST is the 'hello world' of computer vision — 70,000 handwritten digits, studied since 1998, solved by every major architecture. The point of this project is not the dataset. It is the interactive demo: draw a digit, get a prediction. This is the most immediately understandable ML demo there is, and it requires explaining nothing to anyone who sees it.

Under the hood it is a convolutional neural network with two conv blocks, batch normalisation, dropout regularisation, and a softmax output layer. The architecture is classical but the implementation is deliberate — every layer choice is documented.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Flatten,
    Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

from config import (
    MODEL_DIR, PLOTS_DIR, RANDOM_STATE,
    IMG_SIZE, NUM_CLASSES, INPUT_SHAPE,
    EPOCHS, BATCH_SIZE, VALIDATION_SPLIT
)

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
MODEL_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)
print(f"TensorFlow: {tf.__version__}")

## MNIST dataset

60,000 training images, 10,000 test images, 28×28 grayscale pixels, 10 classes.

The dataset is approximately balanced — each digit appears roughly 6,000 times in training. This means accuracy is a reliable metric here, unlike the imbalanced datasets in the credit risk and spam classifier projects.

In [ ]:
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

# Add channel dimension and normalise
X_train = X_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test  = X_test_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0

y_train = to_categorical(y_train_raw, NUM_CLASSES)
y_test  = to_categorical(y_test_raw,  NUM_CLASSES)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")

# Sample digits — one per class
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.flat):
    idx = np.where(y_train_raw == digit)[0][0]
    ax.imshow(X_train_raw[idx], cmap='gray')
    ax.set_title(f'Digit: {digit}')
    ax.axis('off')
fig.suptitle('Sample Digits — One per Class', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_sample_digits.png', dpi=150, bbox_inches='tight')
plt.show()

# Pixel distribution
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(X_train_raw.flatten(), bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
ax.set_xlabel('Pixel intensity (0–255)')
ax.set_ylabel('Frequency')
ax.set_title('Pixel Intensity Distribution — Training Set', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_pixel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Class distribution
train_counts = [np.sum(y_train_raw == d) for d in range(NUM_CLASSES)]
test_counts  = [np.sum(y_test_raw  == d) for d in range(NUM_CLASSES)]
x = np.arange(NUM_CLASSES)
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(x - 0.2, train_counts, 0.4, label='Train', color='steelblue')
ax.bar(x + 0.2, test_counts,  0.4, label='Test',  color='coral')
ax.set_xticks(x)
ax.set_xticklabels([str(d) for d in range(NUM_CLASSES)])
ax.set_xlabel('Digit')
ax.set_ylabel('Count')
ax.set_title('Class Distribution — MNIST is Approximately Balanced', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Why convolutions for images

A fully connected network treats each pixel independently — it has no concept of spatial structure. A 7 drawn in the top-left corner and a 7 drawn in the centre are completely different inputs to a dense layer. Convolutions solve this by learning features that are translation-invariant: the same edge detector fires whether the edge is at pixel (3,3) or pixel (15,15). This is why CNNs are the standard architecture for image classification.

## Architecture — layer-by-layer

**Two Conv2D layers per block** — the first layer in each block learns primitive features (horizontal edges, vertical edges, corners). The second layer in the same block combines those primitives into slightly more complex patterns (curves, junctions). Two layers per block is a balance: enough depth to build up representations without collapsing the spatial dimensions too aggressively.

**BatchNormalization** — normalises each layer's output to have mean ≈ 0 and variance ≈ 1 before passing to the next layer. Without it, the distribution of activations shifts as weights update, forcing each layer to constantly readapt to its input. BatchNorm decouples layer learning, allowing higher learning rates and faster convergence.

**MaxPooling** — takes the maximum value in each 2×2 region, halving the spatial dimensions. The maximum represents the strongest activation of a feature in that region. This gives the model some tolerance to small translations: a stroke shifted by one pixel still produces a similar maximum in the pooling window.

**Dropout** — during training, randomly zeroes a fraction of neurons at each forward pass. This prevents neurons from co-adapting — learning to rely on specific other neurons rather than developing independent features. At inference, all neurons are active and their outputs are scaled down by the dropout rate. The 0.5 rate before the final Dense layer is heavier because dense layers have the most parameters and are the most prone to overfitting.

**Softmax output** — the final Dense(10) layer with softmax activation produces a probability distribution over the 10 digit classes. The probabilities sum to 1, which makes them directly interpretable as confidence scores and enables categorical cross-entropy as the loss function.

In [ ]:
model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', input_shape=INPUT_SHAPE, padding='same'),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Classifier
    Flatten(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint(MODEL_DIR / 'cnn_best.keras', save_best_only=True, monitor='val_accuracy'),
]

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=callbacks,
    verbose=1
)

best_epoch = int(np.argmax(history.history['val_accuracy'])) + 1
print(f"Best epoch: {best_epoch}")

# Training curves
n_epochs_run = len(history.history['loss'])
epochs_range = range(1, n_epochs_run + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_range, history.history['loss'],     label='Train')
ax1.plot(epochs_range, history.history['val_loss'], label='Val')
ax1.axvline(x=best_epoch, color='red', linestyle='--', alpha=0.7, label='Best epoch')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend()
ax2.plot(epochs_range, history.history['accuracy'],     label='Train')
ax2.plot(epochs_range, history.history['val_accuracy'], label='Val')
ax2.axvline(x=best_epoch, color='red', linestyle='--', alpha=0.7, label='Best epoch')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy'); ax2.legend()
fig.suptitle('CNN Training Curves — MNIST', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '04_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Evaluation

The model achieves >99% test accuracy on MNIST — this is expected and not the interesting result. The interesting question is where it fails. The confusion matrix and misclassified examples reveal the genuinely hard cases: 4s that look like 9s, 7s without crossbars that look like 1s, 3s with closed tops that look like 8s. These are the same ambiguities that confuse humans.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc*100:.2f}%")
print(f"Test loss:     {test_loss:.4f}")
print(f"Baseline (random): 10.0% — CNN improvement: {test_acc*100:.1f}%")

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_test_raw, y_pred, target_names=[str(d) for d in range(NUM_CLASSES)]))

cm = confusion_matrix(y_test_raw, y_pred)

# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(range(NUM_CLASSES)); ax.set_yticklabels(range(NUM_CLASSES))
ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
ax.set_title('Confusion Matrix — Test Set (10,000 images)', fontweight='bold')
thresh = cm.max() / 2.0
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > thresh else 'black', fontsize=7)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Misclassified examples
misclassified_idx = np.where(y_pred != y_test_raw)[0]
print(f"\nTotal misclassified: {len(misclassified_idx)} / {len(y_test_raw)}")

fig, axes = plt.subplots(4, 4, figsize=(9, 9))
for i, ax in enumerate(axes.flat):
    if i < len(misclassified_idx):
        idx = misclassified_idx[i]
        ax.imshow(X_test_raw[idx], cmap='gray')
        true_label = y_test_raw[idx]
        pred_label = y_pred[idx]
        conf = float(y_pred_probs[idx][pred_label])
        ax.set_title(f'True:{true_label} Pred:{pred_label}\n{conf*100:.0f}%', fontsize=8)
    ax.axis('off')
fig.suptitle('Misclassified Examples', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '06_misclassified.png', dpi=150, bbox_inches='tight')
plt.show()

# Most confused pairs
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
flat_idx = np.argsort(cm_no_diag.flatten())[::-1][:5]
print("\nMost confused digit pairs:")
for idx in flat_idx:
    true_d = idx // NUM_CLASSES
    pred_d = idx %  NUM_CLASSES
    print(f"  True {true_d} → Predicted {pred_d}: {cm_no_diag[true_d, pred_d]} times")

## The canvas preprocessing challenge

The model was trained on white digits on black background (MNIST convention). The canvas draws black digits on white background (human convention). Without inverting the pixel values before inference, every prediction would be wrong.

This is a subtle but critical preprocessing step — the model is correct, the input format is different. The solution: after normalising canvas pixels to [0,1], apply `image_array = 1.0 - image_array` before passing to the model.

This is the kind of silent failure that breaks deployed models in production when training and inference preprocessing pipelines diverge. There is no exception, no warning — just confident wrong predictions on every input. The fix is one line but the diagnosis requires understanding both the training data format and the inference input format.

## Connecting to the portfolio

This project proves TF/Keras on a third architecture alongside MobileNetV2 (CNN food classifier — transfer learning) and LSTM (stock trend, IMDb sentiment). The three architectures cover the main deep learning paradigms:

- **CNN**: spatial pattern recognition (images)
- **LSTM**: temporal pattern recognition (sequences)
- **CNN + transfer learning**: adapting pretrained features to new domains

Together they demonstrate that the TF/Keras skills claimed in the CV are not limited to one project or one architecture.

## FastAPI vs Flask

The canvas sends a base64-encoded PNG via `fetch()` to `POST /predict`. FastAPI handles this more cleanly than Flask — Pydantic validates the request body automatically, the response schema is documented in `/docs`, and the ASGI architecture handles concurrent prediction requests without blocking.

The consistency with credit-risk-scorer and cv-gap-analyser also matters: using FastAPI across multiple projects signals a deliberate choice, not a one-off experiment.

## Azure App Service Deployment

The trained model (`cnn_best.keras`) is committed to `models/` so Azure App Service loads it without retraining. TensorFlow is the heaviest dependency — the Docker image is large but startup is fast because the model loads in under 2 seconds. The canvas-to-prediction round trip on Azure's F1 free tier is typically 150–300ms — fast enough to feel real-time.

```bash
az group create --name digit-recognizer-rg --location westeurope
az appservice plan create --name digit-recognizer-plan --resource-group digit-recognizer-rg --sku B1 --is-linux
# Scale to F1 via portal after creation
az webapp create --name digit-recognizer-xoc --resource-group digit-recognizer-rg --plan digit-recognizer-plan --runtime "PYTHON:3.11"
az webapp config set --name digit-recognizer-xoc --resource-group digit-recognizer-rg --startup-file "gunicorn main:app --workers 1 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000 --timeout 600"
az webapp config appsettings set --name digit-recognizer-xoc --resource-group digit-recognizer-rg --settings SCM_DO_BUILD_DURING_DEPLOYMENT=true
cd digit-recognizer && zip -r deploy.zip . -x "*.git*" -x "venv/*" -x "__pycache__/*" -x "*.ipynb_checkpoints*"
az webapp deployment source config-zip --name digit-recognizer-xoc --resource-group digit-recognizer-rg --src deploy.zip
```

## Save artefacts & key findings

In [ ]:
history_dict = {
    'loss':         [float(v) for v in history.history['loss']],
    'accuracy':     [float(v) for v in history.history['accuracy']],
    'val_loss':     [float(v) for v in history.history['val_loss']],
    'val_accuracy': [float(v) for v in history.history['val_accuracy']],
}
with open(MODEL_DIR / 'training_history.json', 'w') as f:
    json.dump(history_dict, f, indent=2)

report = classification_report(y_test_raw, y_pred, output_dict=True)
metrics_dict = {
    'test_accuracy': float(test_acc),
    'test_loss':     float(test_loss),
    'best_epoch':    best_epoch,
    'per_class':     {str(d): report[str(d)] for d in range(NUM_CLASSES)},
}
with open(MODEL_DIR / 'model_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

print(f"Test accuracy:  {test_acc*100:.2f}%")
print(f"Best epoch:     {best_epoch}")
print(f"Model saved:    {MODEL_DIR / 'cnn_best.keras'}")
print("\nAll artefacts saved. Update README.md and portfolio.yaml results sections.")